# 02 - Feature Engineering

This notebook performs feature engineering on the clickstream data.

In [ ]:
# =============================================================================
# Import Required Libraries and Custom Modules
# =============================================================================
import sys, importlib
sys.path.append("..")

# Import and reload custom modules to pick up code changes during development
import src.load_data
import src.features
importlib.reload(src.load_data)
importlib.reload(src.features)

# Data loading and preprocessing functions
from src.load_data import load_raw_data, standardize_columns

# Prefix-based feature engineering (non-leaky approach)
from src.features import build_prefix_intent_dataset

## Load Data

In [ ]:
# =============================================================================
# Load and Standardize Raw Clickstream Data
# =============================================================================
# Load semicolon-separated CSV from clothing e-commerce store
df = load_raw_data("../data/raw/events.csv")

# Normalize column names and create synthetic event_time
df = standardize_columns(df)

print(f"Loaded {df.shape[0]} click events from {df['session_id'].nunique()} sessions")
df.head(), df.shape

## Create Features

In [ ]:
# =============================================================================
# Build Prefix-Based Intent Features
# =============================================================================
# Parameters for feature extraction:
# - prefix_n: Number of early clicks to use for features (avoids leakage)
# - horizon_k: Future window defining "quick decision" threshold
#
# Target logic: If session completes within (prefix_n + horizon_k) clicks,
# label as 1 (purchase intent), otherwise 0 (extended browsing)

prefix_n = 5        # Observe first 5 clicks only
horizon_k = 5       # Quick decision = done within 10 total clicks

intent_df = build_prefix_intent_dataset(
    df,
    prefix_n=prefix_n,
    horizon_k=horizon_k
)

print(f"Created {intent_df.shape[1]} features for {intent_df.shape[0]} sessions")
intent_df.head()

In [ ]:
# =============================================================================
# Validate Target Balance and Feature Statistics
# =============================================================================
# Check that target is not degenerate (all 0s or all 1s)
# A balanced target indicates the problem is learnable

print("Target distribution:")
print(intent_df["target"].value_counts(normalize=True).rename("share"))
print("\nPrefix feature statistics:")
intent_df.filter(regex="_pfx$").describe().T

## Save Processed Data

In [ ]:
# =============================================================================
# Save Processed Features to CSV
# =============================================================================
# Export intent dataset for use in modeling notebook
# This file contains prefix-based features that avoid data leakage

import os
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/intent_prefix_dataset.csv"
intent_df.to_csv(output_path, index=False)
print(f"Saved {intent_df.shape[0]} sessions to {output_path}")

## Verify Processed Data

In [ ]:
# =============================================================================
# Verify Data Types
# =============================================================================
# Check column types to ensure proper handling in modeling:
# - Numeric features (price_*_pfx, *_nunique_pfx) should be float/int
# - Categorical mode features (*_mode_pfx) should be object/category
intent_df.dtypes

In [ ]:
# =============================================================================
# Feature Summary Statistics
# =============================================================================
# Review distributions, ranges, and identify potential outliers
# Key features to check:
# - price_std_pfx: High values indicate price comparison behavior
# - *_nunique_pfx: High values indicate exploratory browsing
intent_df.describe()